In [1]:
import os
import math
import time
import torch
import pickle
import numpy as np

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.autograd as autograd

import torch.nn.functional as functional
import torch.optim.lr_scheduler as lr_scheduler
from data_generation import *
from utils import *

os.makedirs('./results', exist_ok=True)
torch.set_default_dtype(torch.float64) 
_ = torch.manual_seed(0)

In [2]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
if torch.cuda.is_available():
    torch_device = 'cuda'
else:
    torch_device = 'cpu'
print(f"TORCH DEVICE: {torch_device}")

TORCH DEVICE: cuda


In [4]:
# Multivariate gaussian
mean = [-1.5,1.5]
mean = torch.tensor(mean)
# cov  = torch.eye(2)
cov = torch.tensor([[1.0,0.5],[0.5,1.0]])


In [5]:
# def log_prob(x,mean, scale = 1.):
#     logp = -1.0*torch.log(torch.tensor(2*np.pi)) - ((x[:, 0] - mean[0]) ** 2 + (x[:, 1] - mean[1]) ** 2)/(2 * scale**2)
#     return logp

In [6]:
batch_size = 512
no_training_samples = 25000
no_validation_samples = 5000
no_testing_samples = 5000

dist  = torch.distributions.MultivariateNormal(mean, covariance_matrix = cov,validate_args=False)
data  = dist.sample((no_training_samples + no_validation_samples + no_testing_samples,)).to(device=torch_device)

train_dataloader = torch.utils.data.DataLoader(data[:no_training_samples], batch_size = batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(data[no_training_samples:no_training_samples + no_validation_samples], batch_size = batch_size, shuffle=False)
test_dataloader = torch.utils.data.DataLoader(data[no_training_samples + no_validation_samples:], batch_size = batch_size, shuffle= False)

In [7]:
logpdf = dist.log_prob(data[:25000].cpu())

In [8]:
logpdf.shape

torch.Size([25000])

In [9]:
array = [data[:25000].cpu().detach().numpy(),-logpdf.numpy()]

In [10]:
output = open("Training_data_multivariate_for_one_gaussian_with_diag_cov_with_exact_log_pdf.pkl", 'wb')
pickle.dump(array, output)
output.close()

In [ ]:
plot_samples_2d(data.cpu().detach().numpy(), name='/results/data_plot')

In [11]:
class SimpleNormal:
    def __init__(self, loc, var, device = torch_device):
        self.dist = torch.distributions.normal.Normal(torch.flatten(loc), torch.flatten(var))
        self.shape = loc.shape
        self.device = device
    def log_prob(self, x):
        logp = self.dist.log_prob(x.reshape(x.shape[0], -1)).to(self.device)
        return torch.sum(logp, dim=1)
    def sample_n(self, batch_size):
        x = self.dist.sample((batch_size,)).to(self.device)
        return x.reshape(batch_size, *self.shape)

In [12]:
class Affine_Coupling(nn.Module):
    def __init__(self,input_shape, hidden_dim):
        super(Affine_Coupling, self).__init__()
        self.input_dim = input_shape
        self.hidden_dim = hidden_dim

        self.layer1 = nn.Linear(self.input_dim, self.hidden_dim)
        self.layer2 = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.scale = nn.Linear(self.hidden_dim, self.input_dim)
        self.translation = nn.Linear(self.hidden_dim, self.input_dim)


    def _compute(self, x):
        out = torch.relu(self.layer1(x))
        out = torch.relu(self.layer2(out))
        return out
    
    def forward(self, x):
        ## convert latent space variable to observed variable
        out = self._compute(x)
        s   = torch.tanh(self.scale(out))
        t   = self.translation(out)
        
        return s,t


In [13]:
class RNVP_2D(nn.Module):
    '''
    A RealNVP class for modeling 2 dimensional distributions
    '''
    def __init__(self,num_coupling_layers,input_shape,hidden_dim):
        '''
        initialized with a list of masks. each mask define an affine coupling layer
        '''
        super(RNVP_2D, self).__init__()        
        self.hidden_dim = hidden_dim  
        self.num_coupling_layers = num_coupling_layers
        self.distribution = SimpleNormal(torch.zeros((2,),device = torch_device), torch.ones((2,),device = torch_device), device = torch_device)
        self.layers_list = nn.ModuleList([Affine_Coupling(input_shape,hidden_dim) for i in range(num_coupling_layers)])
        self.masks = torch.tensor(np.array([[0, 1], [1, 0]] * (num_coupling_layers // 2),
                                                            dtype="float32"),requires_grad = False,device = torch_device)
        

    def forward(self, x):
        ## convert latent space variables into observed variables
        ldj  = 0
        logq = self.distribution.log_prob(x)
        for i in range(self.num_coupling_layers):
            s,t = self.layers_list[i](x*self.masks[i])
            y   = self.masks[i]*x + (1-self.masks[i])*(x*torch.exp(s) + t)        
            ldj+= torch.sum((1 - self.masks[i])*s, -1)
            x = y
        logq -= ldj
        return y, logq

    def inverse(self, y):
        ## convert observed variables into latent space variables        
        ldj = 0
        for i in reversed(range(self.num_coupling_layers)):
            s,t = self.layers_list[i](y*self.masks[i])
            x = self.masks[i]*y + (1-self.masks[i])*((y - t)*torch.exp(-s))
            ldj += torch.sum((1 - self.masks[i])*(-s), -1)
            y = x
        logq  = self.distribution.log_prob(x)
        logq += ldj
        return x, logq

    def log_pdf(self,y):
        ldj  = 0
        for i in reversed(range(self.num_coupling_layers)):
            s,t = self.layers_list[i](y*self.masks[i])
            x = self.masks[i]*y + (1-self.masks[i])*((y - t)*torch.exp(-s))
            ldj += torch.sum((1 - self.masks[i])*(-s), -1)
            y = x
        logq  = self.distribution.log_prob(x)
        logq += ldj
        return logq
        
    
    def jacobian(self, x):
        logq = self.log_pdf(x)
        v = torch.zeros_like(logq)
        v[:] = 1.
        dy_dx = autograd.grad(logq, x, grad_outputs=v, retain_graph=True,
                                create_graph=True, allow_unused=True)[0]  # shape [B, N]
        
        jacobian = dy_dx.requires_grad_()
        return jacobian
      
    
    
        
 

In [14]:
def jacobian(f, x):
    """
    Computes the Jacobian
    """
    
    y = f(x)
    v = torch.zeros_like(y,device = torch_device)
    v[:] = 1.
    dy_dx = autograd.grad(y, x, grad_outputs=v, retain_graph=True,
                                create_graph=True, allow_unused=True)[0]  # shape [B, N]
        
    jacobian = dy_dx.requires_grad_().to(torch_device)
    return jacobian

In [15]:
model = RNVP_2D(4,(2),256).to(torch_device)

In [16]:
optimizer = optim.Adam(model.parameters(),lr= 1.e-6)

In [17]:
PATH = "./results/model_without_cond_with_covariance_fkl.pt"

In [19]:
#For resuming the checkpoint:
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [ ]:
epochs = 100

for i in range(epochs):
    model.train()
    forward_loss = 0.
    NLL = 0.
    
    for x in train_dataloader:
        z, logq = model.inverse(x)
        fkl_loss = -torch.mean(logq)  # Forward Kl loss

 
        optimizer.zero_grad()
        fkl_loss.backward()    
        optimizer.step()
        
        forward_loss += fkl_loss.item()*(x.shape[0])
        
    #scheduler.step()
    
    model.eval()
    with torch.no_grad():
        for x in val_dataloader:
            z_, logq = model.inverse(x)
            nll = - torch.mean(logq)
            NLL += nll.item()*(x.shape[0])
            
            
    mean_fkl = forward_loss/no_training_samples
    mean_nll = NLL/no_validation_samples
            
    template = 'Epoch {:3d},FKL loss: {:.4f},Val loss: {:.4f}'
    print (template.format(i+1 , mean_fkl , mean_nll))
    

In [ ]:
model.eval()
z_, logq = model.inverse(data[no_training_samples + no_validation_samples:])
nll = - torch.mean(logq)

print(f'Test NLL: {nll:.4f}')

In [ ]:
torch.mean(dist.log_prob(data[no_training_samples + no_validation_samples:].cpu()))

In [ ]:
torch.save({'model_state_dict': model.state_dict(),'optimizer_state_dict': optimizer.state_dict(),}, PATH)

In [ ]:
torch.mean(dist.log_prob(data[no_training_samples + no_validation_samples:].cpu()))

In [ ]:
_ = torch.manual_seed(1000)

In [ ]:
test_data  = dist.sample((20000,)).to(device=torch_device)

In [ ]:
neglogp = - dist.log_prob(test_data.cpu())

In [ ]:
data_array = [test_data.cpu().detach().numpy(), neglogp.cpu().detach().numpy()]

In [ ]:
output = open("multivariate_data_for_one_gaussian_with_diagonal_cov_with_neg_logp.pkl", 'wb')
pickle.dump(data_array, output)
output.close()

In [20]:
z_ , logprobtest = model.inverse(data[:25000])

In [21]:
neglogp_nf = -logprobtest

In [22]:
data_with_nf = [data[:25000].cpu().detach().numpy(), neglogp_nf.cpu().detach().numpy()]

In [23]:
output = open("Training_data_multivariate_one_gaussian_with_diag_cov_NF_trained.pkl", 'wb')
pickle.dump(data_with_nf, output)
output.close()